In [97]:
%pip install requests beautifulsoup4 lxml selenium pandas webdriver-manager

Note: you may need to restart the kernel to use updated packages.


In [98]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup
# 드라이버 설정
service =  Service(ChromeDriverManager().install() )
driver = webdriver.Chrome(service=service)

# 페이지 열기
driver.get('https://example.com')

html = driver.page_source   # 정적에서 response.text 와 동일
soup = BeautifulSoup(html, 'lxml')
result = soup.select_one('body > div > p:nth-child(3)')
print(result.text)

#종료
driver.quit()

Learn more


# 다나와 자동차 판매량

In [99]:
url = 'https://auto.danawa.com/auto/?Work=record'


from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup
# 드라이버 설정
service =  Service(ChromeDriverManager().install() )
driver = webdriver.Chrome(service=service)

# 페이지 열기
driver.get(url)

html = driver.page_source   # 정적에서 response.text 와 동일
soup = BeautifulSoup(html, 'lxml')
result = soup.select_one('#autodanawa_gridC > div.gridMain > article > main > div > div:nth-child(3) > div.left > table > tbody')
result = result.select('tr')
for tag in result:
    # print(tag.select_one('.num').text)
    print(tag.select_one('a').text.strip(), tag.select_one('.num').text    )

#종료
driver.quit()

기아 42,066
현대 40,066
제네시스 6,942
KGM 3,701
르노코리아 2,000


# 다나와 2025년 1월부터 12월까지 국산 자동차 판매대수 수집

https://auto.danawa.com/auto/?Work=record&Tab=Grand&Month=2025-01-00

In [100]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup
import time

company, amount = [],[]
year,month = [],[]


# 드라이버 설정
service =  Service(ChromeDriverManager().install() )
driver = webdriver.Chrome(service=service)


for _month in range(1,13):
    _year = '2025'
    _url = f'https://auto.danawa.com/auto/?Work=record&Tab=Grand&Month={_year}-{_month:02}-00'   
    

    # 페이지 열기
    driver.get(_url)
    time.sleep(1)  # 페이지 완전 로딩될때까지 기다림.. 대략 1초면 충분하다고 판단

    html = driver.page_source   # 정적에서 response.text 와 동일
    soup = BeautifulSoup(html, 'lxml')
    result = soup.select_one('#autodanawa_gridC > div.gridMain > article > main > div:nth-child(1) > table > tbody')
    result = result.select('tr')
    count = len(result)
    for tag in result:        
        # print(tag.select_one('a').text.strip(), tag.select_one('.num').contents[0] )
        co, am =  tag.select_one('a').text.strip(), tag.select_one('.num').contents[0] 
        company.append(co); amount.append(am)
    year += [_year]*count
    month += [_month]*count
        

#종료
driver.quit()

In [101]:
import pandas as pd
data = {
    'year' : year,
    'month' : month,
    'company' : company,
    'amount' : amount
}
df = pd.DataFrame(data)
df['amount'] = df['amount'].str.replace(',','').astype(int)
df.head()

,year,month,company,amount
0,2025,1,기아,38412
1,2025,1,현대,37230
2,2025,1,제네시스,8824
3,2025,1,르노코리아,2601
4,2025,1,KGM,2300


# mysql 데이터베이스 적재 하기

```
drop database if exists car_db;
create database car_db;
use car_db;
create table sale_car(
	sale_car_id int auto_increment primary key,
	year varchar(10) not null,
	month tinyint not null,
	company varchar(10) not null,
	amount int not null
);
```

```

In [102]:
import mysql.connector
from mysql.connector import Error
from dotenv import load_dotenv
import os
# 환경변수 읽어오기
load_dotenv()

# 환경변수에서 MySQL 연결 정보 로드
DB_CONFIG = {
    'host': os.getenv('DB_HOST', 'localhost'),
    'user': os.getenv('DB_USER', 'root'),
    'password': os.getenv('DB_PASSWORD', ''),
    'database': os.getenv('DB_NAME', 'car_db'),
    'port': int(os.getenv('DB_PORT', 3306))
}

conn = mysql.connector.connect(**DB_CONFIG)
cursor = conn.cursor()
query = '''
insert into sale_car(year,month,company,amount) 
values(%s,%s,%s,%s)
'''

for _, row in df.iterrows():
    params = (row.year, row.month,row.company, row.amount)
    cursor.execute(query,params=params)

conn.commit()

cursor.close()
conn.close()
print('데이터 삽입 완료')

데이터 삽입 완료


In [103]:
for _, row in df.iterrows():
    print(row.year, row.month,row.company, row.amount)

2025 1 기아 38412
2025 1 현대 37230
2025 1 제네시스 8824
2025 1 르노코리아 2601
2025 1 KGM 2300
2025 1 쉐보레 1219
2025 2 현대 46993
2025 2 기아 46046
2025 2 제네시스 10223
2025 2 르노코리아 4881
2025 2 KGM 2676
2025 2 쉐보레 1453
2025 3 현대 52500
2025 3 기아 50105
2025 3 제네시스 10595
2025 3 르노코리아 6116
2025 3 KGM 3208
2025 3 쉐보레 1372
2025 4 현대 55805
2025 4 기아 51085
2025 4 제네시스 11504
2025 4 르노코리아 5252
2025 4 KGM 3546
2025 4 쉐보레 1298
2025 5 현대 49449
2025 5 기아 45125
2025 5 제네시스 9517
2025 5 르노코리아 4202
2025 5 KGM 3560
2025 5 쉐보레 1383
2025 6 현대 51610
2025 6 기아 46325
2025 6 제네시스 10454
2025 6 르노코리아 5013
2025 6 KGM 3041
2025 6 쉐보레 1266
2025 7 현대 48000
2025 7 기아 45133
2025 7 제네시스 8227
2025 7 KGM 4456
2025 7 르노코리아 4000
2025 7 쉐보레 1205
2025 8 현대 49019
2025 8 기아 43675
2025 8 제네시스 9311
2025 8 KGM 4055
2025 8 르노코리아 3868
2025 8 쉐보레 1181
2025 9 현대 56463
2025 9 기아 49201
2025 9 제네시스 9538
2025 9 르노코리아 4182
2025 9 KGM 4100
2025 9 쉐보레 1208
2025 10 현대 44762
2025 10 기아 40344
2025 10 제네시스 9060
2025 10 르노코리아 3810
2025 10 KGM 3537
2025 10 쉐보레 1172


# FAQ element 요소에 이벤트 추가

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup
import time

url = 'https://www.hyundai.com/kr/ko/faq.html'

# 드라이버 설정
service =  Service(ChromeDriverManager().install() )
driver = webdriver.Chrome(service=service)
driver.get(url)
time.sleep(1)  # 로딩이 될때까지 기다림

html = driver.page_source  # html string

# beautifulsoup 객체를 통해 파싱
soup = BeautifulSoup(html, 'lxml')

result_area = soup.select_one('div.ui_accordion.acc_01')
dl_lists = result_area.select('dl')

print(dl_lists[0].select_one('.title').text)
print(dl_lists[0].select_one('dd').text.strip())

# 첫페이지 클롤링후  다음페이지는 > 클릭해서 페이지 로딩 된후 추출
# > 클릭 --> 프로그램

next_btn = driver.find_element(By.CSS_SELECTOR, 
                    '#contents > div.faq > div > div.section_white > div > div.result_area > div.ui_paging.plugin-paging-apply > nav > button.navi.next')

# 다음페이지 클릭(버튼태그)  >
next_btn.click()

time.sleep(1)

# selector 드랍박스 콤보박스
btn = driver.find_element(By.ID,'category_depth1')
select = Select(btn)
select.select_by_index(1)
time.sleep(1)

# 확인버튼 #contents > div.faq > div > div.section_white > div > form > fieldset > button
confirm_btn = driver.find_element(By.CSS_SELECTOR,
                    '#contents > div.faq > div > div.section_white > div > form > fieldset > button')
confirm_btn.click()

# 검색어 입력하고 조회하기
search = driver.find_element(By.CSS_SELECTOR, 
                    '#contents > div.faq > div > div.section.pd_sm.centered > form > fieldset > div > input')
search.send_keys('차량')
# 검색버튼 클릭
search_btn = driver.find_element(By.CSS_SELECTOR, 
                    '#contents > div.faq > div > div.section.pd_sm.centered > form > fieldset > div > button.btn_search')
search_btn.click()

time.sleep(5)

# 브라우져 닫기
driver.close()


[ 모젠서비스 > 이용단말 ]내비게이션의 지도 업그레이드는 어떻게 하나요?
모젠 지도 업그레이드는 차량용 A/V 및 내비게이션 A/S를 담당하고 있는현대 웰슨에서 수행하고 있습니다. 전국 현대자동차 23개 직영서비스 센터와 기아자동차 11개 직영서비스 센터에서현대 웰슨 A/S요원이 상주 근무하고 있으며전국 115개소의 현대 웰슨 협력점에서 지도 업그레이드가 가능합니다. 모젠고객센터 1588-8111로 전화주시면 지도 업그레이드 상담이 가능하며업그레이드 가능한 가까운 서비스센터에 대한 안내를 받으실 수 있습니다.
